# Phase 1 — Lesson 05: Chain Rule & Automatic Differentiation
> Rich, visual notes for every lesson. Run the code cells to see the concepts in action.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 05
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lesson 04

Every cell below is intentional: explanations first, visualizations second, code third.


# What You'll Learn

---

## Learning Objectives

- Build a minimal autograd engine (Value class) that records operations and computes gradients via reverse-mode autodiff


# The Chain Rule Is the Engine of Learning

---

If `y = f(g(x))`, then:

```
dy/dx = f'(g(x)) × g'(x)
```

Multiply all local derivatives together — this is how gradients flow backward through a network, from loss to weights.


# Computational Graph

---

Every operation is a node:
- **Inputs** = leaves
- **Operations** (add, multiply) = intermediate nodes
- **Output** = root

**Forward pass:** compute values leaves → root.
**Backward pass (reverse-mode autodiff):** compute gradients root → leaves.


In [ ]:
class Value:    def __init__(self, data, _children=(), _op=''):        self.data = data        self.grad = 0.0        self._backward = lambda: None        self.prev = set(_children)        self._op = _op    def __add__(self, other):        other = other if isinstance(other, Value) else Value(other)        out = Value(self.data + other.data, (self, other), '+')        def _backward():            self.grad += out.grad            other.grad += out.grad        out._backward = _backward        return out    def __mul__(self, other):        other = other if isinstance(other, Value) else Value(other)        out = Value(self.data * other.data, (self, other), '*')        def _backward():            self.grad += other.data * out.grad            other.grad += self.data * out.grad        out._backward = _backward        return out    def backward(self):        topo = []; visited = set()        def build_topo(v):            if v not in visited:                visited.add(v)                for child in v.prev: build_topo(child)                topo.append(v)        build_topo(self)        self.grad = 1.0        for node in reversed(topo): node._backward()

In [ ]:
# f(x, y) = (x + y) * xx = Value(2.0); y = Value(-3.0)z = (x + y) * xz.backward()print(f"x.grad = {x.grad}")  # dz/dx = 2x + y = 1print(f"y.grad = {y.grad}")  # dz/dy = x   = 2

# Local Derivatives

---

Each operation contributes its own local derivative in the backward pass:
- **Add:** z = x + y  →  dz/dx = 1, dz/dy = 1 (gradient passes through)
- **Multiply:** z = x·y  →  dz/dx = y, dz/dy = x (gradient scaled by the other input)


# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What is reverse-mode autodiff?
> **Explanation:** A backward pass through the computational graph computing gradients from output to inputs.

**Q2:** For z = x·y, what is dz/dx?
> **Explanation:** y — the gradient w.r.t. x is scaled by the other input.

**Q3:** Why autodiff matters
> **Explanation:** You define the forward computation; gradients are computed automatically (PyTorch, TensorFlow, JAX do this).



# Practice Exercises

---

Do these in order:

**Exercise 1:** Extend Value with `__sub__`, `__truediv__`, and `__pow__` including their `_backward` rules.

**Exercise 2:** Add `exp()` and `log()` operations with correct local derivatives.

**Exercise 3:** Verify autograd gradients match numerical gradients to ~1e-6 for a 3-layer expression.



# Key Terms

---

| Term | What it means |
|------|---------------|
| Chain rule | dy/dx = f'(g(x))·g'(x); multiply local derivatives through nested functions. |
| Computational graph | A DAG where nodes are operations and edges are data flow. |
| Reverse-mode autodiff | Backward pass — efficient for many inputs, one output. |
| Topological sort | Order nodes so dependencies come before dependents. |


# Next Steps

---

When you're ready, continue to Lesson 06: Probability and Distributions.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
